In [0]:
from pyspark.sql.functions import col, trim, regexp_replace
from pyspark.sql.types import *
from pyspark.sql.window import Window

---

# **Silver Data Transformations: Bridge Tables**

Read the bronze bridge tables (`netflix_cast`, `netflix_category`, `netflix_countries`, `netflix_directors`) and load them into silver using the `BridgeLoader` class.

**Cleaning steps**

- Cast `show_id` to `int` (invalid values become NULL).
- Filter out rows where `show_id` is NULL.
- Drop the `_rescued_data` column, as it is no longer needed downstream.
- Trim whitespace and trailing commas from the value column.
- Filter out rows where the value column is NULL or empty.
- Drop duplicate rows.

**Loading (insert-only MERGE)**

- **First run:** write the cleaned data to the silver container and register it as an external Delta table in `netflix_catalog.silver`.
- **Later runs:** run a Delta `MERGE` on `show_id` and the value column (`cast`, `listed_in`, `country`, `director`). New pairs are inserted and existing pairs are skipped, so reruns are idempotent.

In [0]:
BRONZE = "abfss://bronze@netflixprojectetedlll.dfs.core.windows.net"
SILVER = "abfss://silver@netflixprojectetedlll.dfs.core.windows.net"

In [0]:
class BridgeLoader:
    def __init__(self, name, value_col):
        """
        Initializes a BridgeLoader object with the given name and value column.
        The name is used to construct the table and path names, and the value column is used to construct the SQL query.
        """
        
        self.name = name                                # e.g: netflix_cast
        self.value_col = value_col                      # e.g: cast
        self.table = f"netflix_catalog.silver.{name}"   # e.g: netflix_cast
        self.path = f"{SILVER}/{name}"                  # e.g: abfss://silver@netflixprojectetedlll.dfs.core.windows.net/netflix_cast

    def read_and_clean(self):
        """Reads the bronze table, cleans it, and returns a Spark DataFrame"""

        df = spark.read.format("delta").load(f"{BRONZE}/{self.name}") \
                        .withColumn("show_id", col("show_id").try_cast("int")) \
                        .withColumn(self.value_col, trim(regexp_replace(col(self.value_col), r"[,\s]+$", ""))) \
                        .drop("_rescued_data") \
                        .filter(col("show_id").isNotNull()) \
                        .filter(col(self.value_col).isNotNull() & (col(self.value_col) != "")) \
                        .dropDuplicates()

        df.createOrReplaceTempView(f"{self.name}_src")
        return df

    def run(self):

        df = self.read_and_clean()

        if spark.catalog.tableExists(self.table):
            spark.sql(f"""
                MERGE INTO {self.table} AS trg
                USING {self.name}_src AS src
                ON (trg.show_id = src.show_id) AND (trg.{self.value_col} = src.{self.value_col})
                WHEN NOT MATCHED THEN INSERT *
            """)

            print(f"Merged into {self.table}")

        else:
            df.write.format("delta") \
                    .mode("overwrite") \
                    .save(self.path)
                    
            spark.sql(f"CREATE TABLE {self.table} USING DELTA LOCATION '{self.path}'")
            
            print(f"Created {self.table}")

In [0]:
# Run the following to create the silver tables
tables = [
    ("netflix_cast",      "cast"),
    ("netflix_category",  "listed_in"),
    ("netflix_countries", "country"),
    ("netflix_directors", "director"),
]

for name, value_col in tables:
    BridgeLoader(name, value_col).run()

---